# 字符串的模式匹配

本文档部分正则表达式语法解释来源于文献$[1]$. 

参考文献: 
```
[1] Friedl J. Mastering Regular Expressions[M]. 3rd ed. 
    Sebastopol: O’Reilly Media, Inc., 2006.

```

In [1]:
import re; 

In [2]:
motto = "Verba volant, scripta manent"; 

In [3]:
#构造Unicode从U+0000到U+FFFF的所有字符顺次连接组成的字符串
char = str().join( 
    chr(x) for x in range(65536)
); 

在`ipython`命令行或者`Jupyter notebook`中, 可以直接使用特定的转义字符, 改变打印到`sys.stdout`或者笔记本输出单元的字符样式. 本文档据此构造函数, 实现正则表达式匹配内容标记功能. 

In [4]:
#构造生成器, 用于从可迭代对象中有重叠顺次获取相邻两个元素
import copy;  
def adjecent(iter_0): 
    iter_1 = copy.copy(iter_0).__iter__(); 
    iter_2 = copy.copy(iter_0).__iter__(); 
    iter_2.__next__(); 
    for elem_2 in iter_2: 
        elem_1 = iter_1.__next__(); 
        yield elem_1, elem_2; 

In [5]:
#计算正则表达式regex在字符串text中匹配所得子串的起止位置
def pattern_position(text, regex): 
    indices = tuple(
        match.span() for match in re.compile(regex).finditer(text)
    ); 
    return(indices); 

In [6]:
#将正则表达式regex在字符串text中匹配所得子串分布范围使用转义序列标识, 
#以便使用io.TextIOWrapper.write方法打印到stdout或笔记本输出单元
def pattern_highlight(text, regex): 
    indices = pattern_position(text, regex); 
    text_disp = str(); 
    substr_stat = bytearray(len(text) + 1); 
    for idx in indices: 
        start, end = idx; 
        substr_stat[start + 1: end + 1] = (1, ) * (end - start); 
    substr_stat = bytes(substr_stat); 
    for (ch, (start, end)) in zip("\x00" + text, adjecent(substr_stat)): 
        text_disp += ch; 
        if not bool(start) and bool(end): 
            text_disp += "\x1b[07m"; 
        elif bool(start) and not bool(end): 
            text_disp += "\x1b[0m"; 
    text_disp = "\x1b[0m{raw:s}{end:s}\x1b[0m".format(
        raw=text_disp[1:], end=text[-1]
    ); 
    return(text_disp); 

In [7]:
#正则表达式及其匹配结果的对比显示
def pattern_match_illustrate(text, patterns): 
    for regex in [str(), ] + patterns: 
        print("{regex:\x20<24s}{disp:<s}".format(
            regex=regex[: 23], disp=pattern_highlight(text, regex)
        ) )

## 正则表达式语法

In [8]:
hex(ord("|"))

'0x7c'

### 单字符匹配语法
|模式|功能|备注|
|:-|:-:|:-|
|`a`|字面意义上匹配普通字符`a`|以下字符需要使用反斜杠(`\`)转义: <br>`(`, `)`, `[`, `]`, `\`, `.`, `^`, ` `, <br>`*`, `+`, `?`, <code>&#124;</code>|
|`.`|匹配除`\n`(换行符)以外的<br>任何字符|当启用`re.DOTALL`时, 解除对<br>`\n`的匹配限制|
|`[abc]`|匹配`a`, `b`, `c`之一|在方括号中, `-`  (半角负号) 作为<br>待匹配的字符时, 需要转义为`\-`|
|`[^abc]`|匹配除`a`, `b`, `c`以外的任何字符||
|`[a-z]`|匹配字符编码满足 (大于等于`a`且<br>小于等于`z`) 的字符||
|`[^a-z]`|匹配字符编码不满足 (大于等于`a`<br>且小于等于`z`) 的字符||
|`\w`|匹配`_` (半角下划线) , 或者<br>调用`isalnum`方法返回<br>`True`的单个字符||
|`\W`|匹配调用`isalnum`方法返回<br>`False`的单个字符, 不包括`_`||
|`\d`|匹配调用`isdecimal`方法返回<br>`True`的单个字符||
|`\D`|匹配调用`isdecimal`方法返回<br>`False`的单个字符||

* `[abc]`, `[^abc]`, `[a-z]`, `[^a-z]`在同一正则表达式中多次出现时, 不同位置的匹配是相互独立的, \
    例如`[ab][ab]`可以匹配`aa`, `ab`, `ba`或`bb`
* `\w`, `\W`, `\d`, `\D`均可作为方括号中的单字符通配符

In [9]:
tp_regex = [
    r"a", r".", 
    r"[aeiou]", r"[^aeiou]", r"[p-t]", r"[^p-t]", 
    r"\w", r"\W", r"[^aeiou\W]"
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
a                       Verba volant, scripta manent
.                       Verba volant, scripta manent
[aeiou]                 Verba volant, scripta manent
[^aeiou]                Verba volant, scripta manent
[p-t]                   Verba volant, scripta manent
[^p-t]                  Verba volant, scripta manent
\w                      Verba volant, scripta manent
\W                      Verba volant, scripta manent
[^aeiou\W]              Verba volant, scripta manent


#### `\w`, `\d`的匹配范围演示

In [10]:
#表示文本或者数目的字符
str_alnum = str().join(
    [ch for (x, ch) in enumerate(char) if ch.isalnum()]
); 

In [11]:
#正则表达式\w语法可以匹配的字符
re_mtch_word = str().join(re.compile("\w").findall(char)); 

In [12]:
for prop in "issuperset", "__eq__", "issubset": 
    print("{rel:\x20<12s}{propos!s}".format(
        rel=prop, 
        propos=set(re_mtch_word).__getattribute__(prop)(set(str_alnum))
    ) )

issuperset  True
__eq__      False
issubset    False


In [13]:
#正则表达式\w语法可以匹配的字符, 比满足isalnum的字符范围多一个_ (半角下划线)
set(re_mtch_word).difference(set(str_alnum))

{'_'}

In [14]:
#可用于十进制数码的字符
str_dec = str().join(
    [ch for (x, ch) in enumerate(char) if ch.isdecimal()]
); 

In [15]:
#正则表达式\d语法可以匹配的字符
re_mtch_dec = str().join(re.compile("\d").findall(char)); 

In [16]:
for prop in "issuperset", "__eq__", "issubset": 
    print("{rel:\x20<12s}{propos!s}".format(
        rel=prop, 
        propos=set(re_mtch_dec).__getattribute__(prop)(set(str_dec))
    ) )

issuperset  True
__eq__      True
issubset    True


### 多字符匹配语法
|模式|功能|备注|
|:-|:-:|:-|
|`a?`|采用**贪心策略**匹配零个或<br>一个字符`a`|含该语法的模式会在符合匹配<br>规则的前提下, 匹配**最长**的子串|
|`a+`|采用贪心策略匹配一个<br>或**连续多个**字符`a`||
|`a*`|采用贪心策略匹配零个, <br>一个或**连续多个**字符`a`||
|`a{m}`|采用贪心策略匹配**连续<br>`m`个**字符`a`||
|`a{m,n}`|采用贪心策略匹配**连续<br>多个**字符`a`, 最少`m`个, <br>最多`n`个|`m`缺省时为`0`, `n`缺省时<br>为无穷大, 但逗号不可省略|
|`a??`, <br>`a+?`, <br>`a*?`, <br>`a{m}?`, <br>`a{m,n}?`|采用**懒惰策略**匹配特定<br>数量的字符`a`|含该语法的模式会在符合匹配<br>规则的前提下, 匹配**最短**的子串|

* `a?`, `a+`, `a*`, `a{m}`均可视为`a{m,n}`的特例, 分别与`a{,1}`, `a{1,}`, `a{,}`, `a{m,m}`等效. 
* 上述任何语法在同一正则表达式中多次出现时, 不同位置的匹配是相互独立的, \
    例如`a?b?c`可以匹配`c`, `ac`, `bc`或`abc` 

In [17]:
tp_regex = [
    r"a\w?nt", r"a\w+nt", r"a\w*nt", 
    r"[^aeiou\W]{2}", r"[^aeiou\W]{2,}", r"[^aeiou\W]{,2}", 
    r"a.+a", r"a.+?a"
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
a\w?nt                  Verba volant, scripta manent
a\w+nt                  Verba volant, scripta manent
a\w*nt                  Verba volant, scripta manent
[^aeiou\W]{2}           Verba volant, scripta manent
[^aeiou\W]{2,}          Verba volant, scripta manent
[^aeiou\W]{,2}          Verba volant, scripta manent
a.+a                    Verba volant, scripta manent
a.+?a                   Verba volant, scripta manent


### 字符组合语法
|模式|功能|备注|
|:-|:-:|:-|
|`(abc)`|字符`abc`顺次构成的子串作为整体, <br>构成一个**匿名编组**参与匹配||
|`(?:abc)`|字符`abc`顺次构成的子串作为整体<br>参与匹配, 但**不构成**编组||
|`(?P<idx>ab)`|字符`ab`构成**实名编组**参与<br>匹配, 编组代号为`idx`|`idx`的内容需为合法的<br>python关键字标识符|
|<code>ab&#124;cd</code>|匹配`ab`或`cd`|匹配范围的界限是未经<br>转义的以下字符: <br><code>&#124;</code>左侧的未成对`(` <br>或<code>&#124;</code>右侧的未成对`)`<br>或<code>&#124;</code>|
|`(?P=id)`|同一正则表达式中, 代号为`id`的<br>实名编组重新调用|`(?P=id)`必须位于<br>`(?P<id>ab)`之后, 中间<br>允许插入其他正则表达式<br>语法, 或者直接相邻; |

* 前述的[多字符匹配语法](#多字符匹配语法)也适用于编组, 匹配的范围是**将编组作为整体后**重复特定次数的子串, \
    例如`(ab){1,3}`可以匹配`ab`, `abab`, `ababab`
* **同一次匹配中**, 同一代号的编组在**定义位置`(?P<id>ab)`和所有调用位置`(?P=id)`**匹配的字符串内容**完全相同**, 可用于**取消通配符匹配的独立性**, \
    例如`(?P<same>[ab])(?P=same)`只能匹配`aa`和`bb`, 不能匹配`ab`和`ba`, \
    而`[ab][ab]`和`[ab]{2}`均可匹配`aa`, `ab`, `ba`或`bb`

In [18]:
tp_regex = [
    r"[^aeiou\W][aeiou]{2,}", r"([^aeiou\W][aeiou]){2,}", 
    r"an|en", r"an|en(t)",  r"an(|en)t",  r"(an|en)t",  r"(an(|e))nt", 
    r"[aeiou].*?[aeiou]", r"(?P<v>[aeiou]).*?(?P=v)"
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
[^aeiou\W][aeiou]{2,}   Verba volant, scripta manent
([^aeiou\W][aeiou]){2,} Verba volant, scripta manent
an|en                   Verba volant, scripta manent
an|en(t)                Verba volant, scripta manent
an(|en)t                Verba volant, scripta manent
(an|en)t                Verba volant, scripta manent
(an(|e))nt              Verba volant, scripta manent
[aeiou].*?[aeiou]       Verba volant, scripta manent
(?P<v>[aeiou]).*?(?P=v) Verba volant, scripta manent


### 字符定位语法
|格式|功能|备注|
|:-|:-:|:-|
|`abc$` |匹配模式`abc`, 要求其结尾<br>必须为字符串结尾|等效形式: `abc\Z`|
|`abc(?=def)`|匹配模式`abc`, 要求**其后**<br>紧随模式`def`||
|`abc(?!def)`|匹配模式`abc`, 要求其后<br>紧随的内容不允许匹配模式`def`||
|`^abc`|匹配模式`abc`, 要求其开头<br>必须为字符串开头|等效形式: `\Aabc`|
|`(?<=def)abc`|匹配模式`abc`, 要求**其前**<br>紧邻模式`def`|模式`def`匹配的子串<br>**宽度必须为固定值**|
|`(?<!def)abc`|匹配模式`abc`, 要求其前<br>紧邻的内容不允许匹配模式`def`|模式`def`匹配的子串<br>**宽度必须为固定值**|

In [19]:
tp_regex = [
    r"[^aeiou\W]", r"[^aeiou\W]$", r"^[^aeiou\W]", 
    r"[aeiou][^aeiou\W]?", r"[aeiou](?=[^aeiou\W])", r"[aeiou](?![^aeiou\W])", 
    r"n?t", r"(?<=n)t", r"(?<!n)t", 
]; 
pattern_match_illustrate(motto, tp_regex)

                        Verba volant, scripta manent
[^aeiou\W]              Verba volant, scripta manent
[^aeiou\W]$             Verba volant, scripta manent
^[^aeiou\W]             Verba volant, scripta manent
[aeiou][^aeiou\W]?      Verba volant, scripta manent
[aeiou](?=[^aeiou\W])   Verba volant, scripta manent
[aeiou](?![^aeiou\W])   Verba volant, scripta manent
n?t                     Verba volant, scripta manent
(?<=n)t                 Verba volant, scripta manent
(?<!n)t                 Verba volant, scripta manent


## 字符串模式匹配结果的定位与提取

### `re.Pattern`对象的构造

为提高批量匹配的效率, 建议在使用正则表达式前, 通过`re.complle`方法编译正则表达式. 

用法: 
```python
patt = re.compile(regex_syntax)
```
* `regex_syntex` 正则表达式语法, `str`对象或`bytes`对象; 
* 返回结果为`re.Pattern`对象; 
* 由`str`对象编译所得的`re.Pattern`对象只能用于匹配`str`; \
    由`bytes`对象编译所得的`re.Pattern`对象只能用于匹配`bytes`
    
注意事项: 
* 文本的提取是形式层面的, 建议仅将正则匹配用于形式检查和匹配, **不要在匹配规则中加入过多的内容判定**功能, 例如数据范围判定, 校验码计算等; 
* 在实际的工程应用中, 将基于正则表达式的形式检查功能, 与内容判定功能分离, 形式检查通过后, 匹配和提取的结果方可传入后续的内容判定工序中, 提高工程的可维护性. 

In [20]:
#中文公历日期格式匹配: YYYY年M月D日, 要求: 
#1. 匹配后可以直接提取年月日, 无需slice
#2. 月份和日期的有效性不作检查
date_ymd_zh_filter = re.compile(
    r"(?P<year>[1-9][0-9]{3})年(?P<month>[0-9]{,2})月(?P<day>[0-9]{,2})日"
); 
covid_quarantine_intvl = "2020年1月23日至2020年4月8日"; 

### 基于`re.Pattern`对象的模式匹配操作
|方法|功能|备注|
|:-|:-:|:-|
|`search(text, pos, endpos)`|从`text`的第`pos`个字符(含)开始, <br>逐个字符向后搜索, 并尝试模式匹配, <br>直至其第`endpos`个字符为止|当模式在`text`中首次匹配成功时, 停止匹配<br>并返回第一个匹配结果的`re.Match`对象; <br>搜索结束后仍无法匹配时, 返回`None`|
|`match(text, pos, endpos)`|从`text`的第`pos`个字符(含)开始<br>模式匹配, 直至其第`endpos`个字符为止, <br>要求匹配的**子串开头必须位于**`text`的<br>第`pos`个字符处. |当模式匹配成功时, 返回匹配结果的<br>`re.Match`对象; 匹配失败时, 返回`None`|
|`fullmatch(text, pos, endpos)`|从`text`的第`pos`个字符(含)开始<br>模式匹配, 直至其第`endpos`个字符为止, <br>要求匹配的**子串开头必须位于**`text`的<br>第`pos`个字符处, **同时结尾必须位于**<br>第`endpos`个字符处. |当模式匹配成功时, 返回匹配结果的<br>`re.Match`对象; 匹配失败时, 返回`None`|
|`finditer(text, pos, endpos)`|从`text`的第`pos`个字符(含)开始, <br>逐个字符向后搜索, 并尝试模式匹配, <br>直至其第`endpos`个字符为止|调用该方法后**立刻返回一个生成器**对象, <br>每次调用该对象的`__next__`方法后, <br>将从**上次匹配结果结尾之后的下一个字符**<br>开始搜索和匹配 (**无重叠**匹配), 如匹配成功, <br>则`yield`对应的`re.Match`对象|

* 所有方法中`pos`和`endpos`均为可选参数, `pos`默认值为`0`, `endpos`默认值为`sys.maxsize`, 在32位解释器为$2^{31} - 1$, 在64位解释器为$2^{63} - 1$

In [21]:
import itertools as it; 
props = "search", "match", "fullmatch", "finditer"; 
pos = 0, 2, 11; 
print(date_ymd_zh_filter.pattern, covid_quarantine_intvl, sep="\n")
for prop, idx in it.product(props, pos): 
    print("{0:11s} pos={1:\x20<3d} {2!s:<s}".format(
        prop, idx, date_ymd_zh_filter.__getattribute__(prop)(
            covid_quarantine_intvl, pos=idx
        )
    ) )

(?P<year>[1-9][0-9]{3})年(?P<month>[0-9]{,2})月(?P<day>[0-9]{,2})日
2020年1月23日至2020年4月8日
search      pos=0   <re.Match object; span=(0, 10), match='2020年1月23日'>
search      pos=2   <re.Match object; span=(11, 20), match='2020年4月8日'>
search      pos=11  <re.Match object; span=(11, 20), match='2020年4月8日'>
match       pos=0   <re.Match object; span=(0, 10), match='2020年1月23日'>
match       pos=2   None
match       pos=11  <re.Match object; span=(11, 20), match='2020年4月8日'>
fullmatch   pos=0   None
fullmatch   pos=2   None
fullmatch   pos=11  <re.Match object; span=(11, 20), match='2020年4月8日'>
finditer    pos=0   <callable_iterator object at 0x0000000005CF4088>
finditer    pos=2   <callable_iterator object at 0x0000000005A90408>
finditer    pos=11  <callable_iterator object at 0x0000000005CF4088>


### 从`re.Match`对象提取匹配结果
|方法|功能|备注|
|:-|:-:|:-|
|`m.string()`|接受匹配的字符串||
|`m.group(0)`|整个正则表达式的匹配结果||
|`m.start(0)`|整个正则表达式匹配结果的<br>开头在接受匹配字符串的位置|从接受匹配字符串的开头起算, <br>与`pos`的值无关|
|`m.end(0)`|整个正则表达式匹配结果的<br>结尾在接受匹配字符串的位置**加1**|从接受匹配字符串的开头起算, <br>与`pos`的值无关|
|`m.span(0)`|整个正则表达式匹配结果<br>在接受匹配字符串中的索引元组|二元`tuple`, 等效于<br>`(m.start(), m.end())`; <br>可以使用`slice(*m.span())`<br>解包转化为索引|
|`m.groupdict()`|正则表达式中所有**实名编组**<br>的匹配结果|返回的`dict`中, 所有的键为正则<br>表达式中所有实名编组的代号; <br>每个键对应的的值为以该键为代号<br>的编组的匹配结果; <br>在同一次匹配中, 如果同一个正则<br>表达式的同一个实名编组被匹配了<br>多次, 则`dict`中对应的值为**最后<br>一次**匹配的结果|

In [22]:
print(date_ymd_zh_filter.pattern, covid_quarantine_intvl, sep="\n"); 
for m in date_ymd_zh_filter.finditer(covid_quarantine_intvl): 
    print(m); 
    for mthd in "group", "span", "groupdict": 
        print("\x20\x20{0:\x20<12s} -> {1!s}".format(
            mthd, m.__getattribute__(mthd)(0)
        ) )

(?P<year>[1-9][0-9]{3})年(?P<month>[0-9]{,2})月(?P<day>[0-9]{,2})日
2020年1月23日至2020年4月8日
<re.Match object; span=(0, 10), match='2020年1月23日'>
  group        -> 2020年1月23日
  span         -> (0, 10)
  groupdict    -> {'year': '2020', 'month': '1', 'day': '23'}
<re.Match object; span=(11, 20), match='2020年4月8日'>
  group        -> 2020年4月8日
  span         -> (11, 20)
  groupdict    -> {'year': '2020', 'month': '4', 'day': '8'}


## 基于模式匹配的字符串替换和拆分

### 字符串替换
用法: 
```python
patt.sub(repl, text, n)
```
从`text`的开头开始, **无重复**, 逐个字符向后搜索, 尝试模式匹配`patt`, 并将匹配到的子串替换为`repl`中的内容, 持续`n`次或搜索至字符串末尾时停止, 返回替换所得的新字符串
* `patt`为经`re.compile`预编译所得的正则表达式, 即`re.Pattern`对象; 
* 当`not patt.search(text)`时, 返回的新字符串与原字符串相同; 
* 当`patt`的正则表达式语法中使用实名编组`(?P<id>xxx)`时, 在`repl`中可使用`\g<id>`表示正则表达式在`text`对应位置匹配过程中, 该编组所匹配的结果; 
* `n`缺省时取`0`, 表示匹配和替换过程持续到字符串末尾 (无重复替换所有匹配的子串)

```python
patt.subn(repl, text, n)
```
从`text`的开头开始, **无重复**, 逐个字符向后搜索, 尝试模式匹配`repl`, 并将匹配到的子串替换为`repl`中的内容, 持续`n`次或搜索至字符串末尾时停止, 返回二元`tuple`
* `patt`, `repl`, `n`的要求与`sub`方法相同; 
* 结果的首个元素为替换所得的新字符串; 
* 结果的末个元素为替换操作**实际执行**的次数

In [23]:
print(date_ymd_zh_filter.pattern, covid_quarantine_intvl, sep="\n"); 
print(date_ymd_zh_filter.sub("某年某月某日", covid_quarantine_intvl)); 
print(date_ymd_zh_filter.sub(
    "\g<year>-\g<month>-\g<day>", covid_quarantine_intvl
) ); 
print(date_ymd_zh_filter.sub(
    "\g<year>-\g<month>-\g<day>", covid_quarantine_intvl, 1
) ); 
print(date_ymd_zh_filter.subn(
    "\g<year>-\g<month>-\g<day>", covid_quarantine_intvl
) ); 

(?P<year>[1-9][0-9]{3})年(?P<month>[0-9]{,2})月(?P<day>[0-9]{,2})日
2020年1月23日至2020年4月8日
某年某月某日至某年某月某日
2020-1-23至2020-4-8
2020-1-23至2020年4月8日
('2020-1-23至2020-4-8', 2)


## 字符串拆分
```python
patt.split(text, n)
```
从`text`的开头开始, **无重复**, 逐个字符向后搜索, 尝试模式匹配`patt`, 将整个正则表达式匹配到的子串视为分隔符, 并将分隔符前后(不含其自身)的部分拆分, 重复`n`次或搜索至字符串末尾时停止, 返回拆分后的子串构成的`list`
* 各子串顺序与其在`text`中顺序相同; 
* 当`not patt.search(text)`时, 返回`[text]`; 
* 当`n`缺省时取`0`, 表示匹配和截断过程持续到字符串末尾

In [24]:
print(re.compile(r"[\W]+").split(motto)); 
print(re.compile(r"[\W]+").split(motto, 2)); 

['Verba', 'volant', 'scripta', 'manent']
['Verba', 'volant', 'scripta manent']
